In [0]:
# Read CSV from Bronze Volume
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/workspace/bronze/practice/people-1000.csv")

df.display()


In [0]:
df = df.toDF(*[
    "index",
    "user_id",
    "first_name",
    "last_name",
    "sex",
    "email",
    "phone",
    "date_of_birth",
    "job_title",
    
])

display(df)

In [0]:
from pyspark.sql import functions as F

# Read CSV from volume (adjust path to yours)
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/workspace/bronze/practice/people-1000.csv")

# Fix column names
new_cols = [c.replace(" ", "_").lower() for c in df.columns]
df = df.toDF(*new_cols)

# Add ingest timestamp — this is your watermark column
df = df.withColumn("ingest_ts", F.current_timestamp())

# Overwrite bronze (Bronze is append-friendly but overwrite is fine for practice)
df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.bronze.bronze_people")

print(f"Bronze rows: {df.count()}")


In [0]:
df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.bronze.bronze_people")


In [0]:
df = spark.read.table("workspace.bronze.bronze_people")
print("Row count:", df.count())
print("Columns:", df.columns)
df.select("user_id", "ingest_ts").show(5)


In [0]:
df_audit = spark.read.table("workspace.audit.people_pipeline_runs")
print("Audit row count:", df_audit.count())
df_audit.show()
